# FairValue Agent Demo

This notebook demonstrates the core FairValue Agent workflow for educational stock analysis. The project uses Yahoo Finance data, simple DCF scenarios, a P/E cross-check, and rule-based risk warnings.

**Disclaimer:** This is not financial advice, an investment recommendation, or a guarantee of future returns.

## 1. Import Project Components

In [ ]:
from fairvalue_agent.agents.data_agent import DataAgent
from fairvalue_agent.agents.discovery_agent import DiscoveryAgent
from fairvalue_agent.agents.fundamental_agent import FundamentalAnalysisAgent
from fairvalue_agent.agents.report_agent import ReportAgent
from fairvalue_agent.agents.risk_agent import RiskAgent
from fairvalue_agent.agents.valuation_agent import ValuationAgent
from fairvalue_agent.tools.formatting_tools import format_money
from fairvalue_agent.tools.market_tools import resolve_yahoo_symbol

## 2. Ticker And Market Discovery

Use this when you know the company, sector, or theme but do not know the exact ticker or market. This cell needs internet access because it calls Yahoo Finance search.

In [ ]:
discovery = DiscoveryAgent().run("HSBA", preferred_market="UK")

for candidate in discovery.candidates[:5]:
    print(candidate.symbol, candidate.market_code, candidate.exchange_name, candidate.name, candidate.sector)

## 3. Market-Aware Symbol Resolution

The app converts local tickers into Yahoo Finance symbols. This part works offline.

In [ ]:
examples = [("AAPL", "US"), ("700", "HK"), ("HSBA", "UK"), ("7203", "JP")]

for ticker, market in examples:
    symbol, market_config = resolve_yahoo_symbol(ticker, market)
    print(f"{ticker} + {market} -> {symbol} ({market_config.label})")

## 4. Run One Full Valuation

This cell needs internet access because it fetches live Yahoo Finance data. Try `AAPL` for US data or `HSBA` with `UK` to test London pence/GBP normalization.

In [ ]:
ticker = "AAPL"
market = "US"

data_result = DataAgent().run(ticker, market=market)
fundamentals = FundamentalAnalysisAgent().run(data_result)
valuation = ValuationAgent().run(data_result)
risk_assessment = RiskAgent().run(data_result, valuation)

company = data_result.company
currency = company.currency or "USD"

print("Company:", company.company_name or company.ticker)
print("Ticker:", company.ticker)
print("Market:", company.market)
print("Current price:", format_money(company.current_price, currency))
print("Fair value range:", format_money(valuation.fair_value_low, currency), "-", format_money(valuation.fair_value_high, currency))
print("Valuation label:", valuation.valuation_label)
print("Fundamental quality:", fundamentals.overall_quality)
print("Overall risk:", risk_assessment.overall_risk)

if valuation.warnings:
    print("\nWarnings:")
    for warning in valuation.warnings:
        print("-", warning)

## 5. Generate A Markdown Report

In [ ]:
report = ReportAgent().run(data_result, fundamentals, valuation, risk_assessment)
print(report.markdown[:1500])

## Notes For Reviewers

- Yahoo Finance data can be delayed, incomplete, or unavailable for some markets.
- The DCF model is intentionally simple and should be treated as a transparent educational approximation.
- London listings are normalized from pence (`GBp`) into GBP, and financial statements are converted into the quote currency when possible.
- This project is designed for learning, not investment recommendations.